# Beca 18 RAG Chatbot

**Course:** Python Programming -- Applied Data Science  
**Task:** Retrieval-Augmented Generation pipeline over the official Beca 18 regulations (PRONABEC).  
**Source document:** Resolucion Directoral Ejecutiva N. 033-2026-MINEDU/VMGI-PRONABEC

Pipeline overview:

```
PDF -> text extraction -> chunking -> embeddings -> ChromaDB
    -> user question -> query embedding -> top-k retrieval
    -> LLM with context -> grounded answer + cited sources
```

## Step 0 -- Setup

Install dependencies and load the Gemini API key from a `.env` file using `python-dotenv`. Never hardcode the key in the notebook. Confirm the environment by printing the loaded package versions.

Get a free API key at: https://aistudio.google.com/app/apikey

In [ ]:
# Uncomment to install in Google Colab (skip locally if conda env already has the packages)
# !pip install pypdf==5.1.0 tiktoken==0.8.0 langchain-text-splitters==0.3.2 google-genai chromadb==0.5.23 ipywidgets==8.1.5 tqdm==4.67.1 python-dotenv==1.0.1

In [1]:
import os
import sys
from pathlib import Path
from importlib.metadata import version

from dotenv import load_dotenv

# --- Load .env from the repo root (parent of notebooks/) ---
ENV_PATH = Path('..').resolve() / '.env'
loaded = load_dotenv(ENV_PATH)

if not loaded:
    raise FileNotFoundError(
        f'.env not found at {ENV_PATH}. '
        'Copy .env.example to .env and add your GEMINI_API_KEY.'
    )

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY or GEMINI_API_KEY == 'your_key_here':
    raise ValueError(
        'GEMINI_API_KEY is missing or still the placeholder. '
        'Edit .env and set a real key from https://aistudio.google.com/app/apikey'
    )

# --- Repository paths ---
REPO_ROOT  = Path('..').resolve()
DATA_DIR   = REPO_ROOT / 'data'
PDF_PATH   = DATA_DIR / 'beca18_reglamento.pdf'
CHROMA_DIR = REPO_ROOT / 'chroma_db_beca18'

if not PDF_PATH.exists():
    raise FileNotFoundError(f'PDF not found at {PDF_PATH}')

# --- Print environment confirmation ---
print('Environment setup OK')
print()
print(f'Python      : {sys.version.split()[0]}')
print(f'Repo root   : {REPO_ROOT}')
print(f'PDF source  : {PDF_PATH.name}  ({PDF_PATH.stat().st_size / 1024:.1f} KB)')
print(f'Chroma path : {CHROMA_DIR}')
print()
print('Loaded API key: GEMINI_API_KEY = ' + GEMINI_API_KEY[:6] + '...' + GEMINI_API_KEY[-4:])
print('  (key is hidden, never printed in full and never hardcoded)')
print()
print('Package versions:')
for pkg in [
    'pypdf', 'tiktoken', 'langchain-text-splitters', 'google-genai',
    'chromadb', 'ipywidgets', 'tqdm', 'python-dotenv',
]:
    try:
        print(f'  {pkg:30s} {version(pkg)}')
    except Exception as e:
        print(f'  {pkg:30s} (not installed: {e})')


Environment setup OK

Python      : 3.11.15
Repo root   : C:\Users\Diego\OneDrive\Documents\GitHub\beca18-rag-chatbot
PDF source  : beca18_reglamento.pdf  (2260.0 KB)
Chroma path : C:\Users\Diego\OneDrive\Documents\GitHub\beca18-rag-chatbot\chroma_db_beca18

Loaded API key: GEMINI_API_KEY = AIzaSy...oGDE
  (key is hidden, never printed in full and never hardcoded)

Package versions:
  pypdf                          5.1.0
  tiktoken                       0.8.0
  langchain-text-splitters       0.3.2
  google-genai                   2.2.0
  chromadb                       0.5.23
  ipywidgets                     8.1.5
  tqdm                           4.67.1
  python-dotenv                  1.0.1


## Step 1 -- PDF Text Extraction

Extract text page by page using `pypdf`. Insert a `[PAGE N]` marker at the start of each page. Apply light cleaning: collapse multiple spaces, remove isolated line breaks, and strip headers/footers if present. Print total character and word counts.

In [2]:
import re
from pypdf import PdfReader

# --- Extract text page by page with markers ---
reader = PdfReader(str(PDF_PATH))
n_pages = len(reader.pages)
print(f'PDF loaded: {n_pages} pages')

raw_pages = []
for i, page in enumerate(reader.pages, start=1):
    txt = page.extract_text() or ''
    raw_pages.append((i, txt))

# --- Light cleaning function ---
def clean_page(text: str) -> str:
    # Strip leading/trailing whitespace
    text = text.strip()

    # Remove common headers/footers (page numbers, copyright lines, etc.)
    # Heuristic: drop very short lines that look like footers/page numbers
    lines = text.split('\n')
    cleaned_lines = []
    for ln in lines:
        s = ln.strip()
        # Drop empty lines and isolated page numbers like "12" or "Pag. 12"
        if not s:
            continue
        if re.fullmatch(r'(p[aá]g(ina)?\.?\s*)?\d{1,3}', s, flags=re.IGNORECASE):
            continue
        cleaned_lines.append(s)

    text = '\n'.join(cleaned_lines)

    # Collapse multiple spaces and tabs
    text = re.sub(r'[ \t]+', ' ', text)

    # Remove isolated line breaks: join lines that don't end with punctuation
    # (keeps paragraph breaks marked by double newlines)
    text = re.sub(r'(?<![.!?:;\n])\n(?!\n)', ' ', text)

    # Collapse 3+ consecutive newlines into 2 (preserve paragraph breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

# --- Apply cleaning and stitch full document with page markers ---
clean_pages = []
for page_num, raw_text in raw_pages:
    cleaned = clean_page(raw_text)
    if cleaned:
        clean_pages.append(f'[PAGE {page_num}]\n{cleaned}')

full_text = '\n\n'.join(clean_pages)

# --- Report ---
total_chars = len(full_text)
total_words = len(full_text.split())

print()
print('=== Extraction Summary ===')
print(f'  Pages processed       : {n_pages}')
print(f'  Pages with content    : {len(clean_pages)}')
print(f'  Total characters      : {total_chars:,}')
print(f'  Total words           : {total_words:,}')
print(f'  Avg chars per page    : {total_chars / max(len(clean_pages), 1):,.0f}')
print()
print('=== Preview (first 600 chars) ===')
print(full_text[:600])
print('...')


PDF loaded: 138 pages

=== Extraction Summary ===
  Pages processed       : 138
  Pages with content    : 138
  Total characters      : 365,895
  Total words           : 55,035
  Avg chars per page    : 2,651

=== Preview (first 600 chars) ===
[PAGE 1]
Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS:
El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y Bienestar; el Informe N° 042-2026-MINEDU/VMGI-PRONABEC-OPP de la Oficina de Planeamiento y Presupuesto; el Informe N ° 048-2026-MINEDU/VMGI-PRONABEC-OAJ de la Oficina de Asesoría Jurídica, y;
CONSIDERANDO:
Que, la Ley N° 29837 crea el Programa Nacional de Becas y Crédito Educativo (en adelante, el PRONABEC), a cargo del Ministerio de Edu
...
